Advanced: Constraints and Changing Hierarchies
============================

Setup
-----------------------------

Let's first make sure we have the latest version of PHOEBE 2.5 installed (uncomment this line if running in an online notebook session such as colab).

In [1]:
#!pip install -I "phoebe>=2.5"

As always, let's do imports and initialize a logger and a new Bundle.

In [2]:
import phoebe
from phoebe import u # units
import numpy as np
import matplotlib.pyplot as plt

logger = phoebe.logger()

b = phoebe.default_binary()

Mon, 22 Jun 2026 14:23 BUNDLE       WARNING importing from an older version (2.4.21) of PHOEBE to PHOEBE 2.5+.  This may take some time.  Please check all values.
Mon, 22 Jun 2026 14:23 BUNDLE       WARNING 'teffratio' not a recognized kwarg
Mon, 22 Jun 2026 14:23 BUNDLE       WARNING 'requivratio' not a recognized kwarg
Mon, 22 Jun 2026 14:23 BUNDLE       WARNING 'requivsumfrac' not a recognized kwarg


Changing Hierarchies
-------------------------------------

Some of the built-in constraints depend on the system hierarchy, and will automatically adjust to reflect changes to the hierarchy.

For example, the masses depend on the period and semi-major axis of the parent orbit but also depend on the mass-ratio (q) which is defined as the secondary mass over primary mass.  For this reason, changing the roles of the primary and secondary components should be reflected in the masses (so long as q remains fixed).

In order to show this example, let's set the mass-ratio to be non-unity.

In [3]:
b.set_value('q', 0.8)

Here the star with component tag 'primary' is actually the primary component in the hierarchy, so should have the LARGER mass (for a q < 1.0).

In [4]:
print("M1: {}, M2: {}".format(b.get_value(qualifier='mass', component='primary', context='component'),
                              b.get_value(qualifier='mass', component='secondary', context='component')))

M1: 1.109792361995535, M2: 0.8878338895964281


Now let's flip the hierarchy so that the star with the 'primary' component tag is actually the secondary component in the system (and so takes the role of numerator in q = M2/M1).

For more information on the syntax for setting hierarchies, see the [Building a System Tutorial](building_a_system.ipynb).

In [5]:
b['mass@primary']

<ParameterSet: 2 parameters | contexts: constraint, component>

In [6]:
b.set_hierarchy('orbit:binary(star:secondary, star:primary)')

In [7]:
b['mass@primary@star@component']

<Parameter: mass=0.8878338895964281 solMass | keys: description, value, quantity, default_unit, limits, visible_if, copy_for, readonly, advanced, latexfmt>

In [8]:
print(b.get_value('q'))

0.8


In [9]:
print("M1: {}, M2: {}".format(b.get_value(qualifier='mass', component='primary', context='component'),
                              b.get_value(qualifier='mass', component='secondary', context='component')))

M1: 0.8878338895964281, M2: 1.109792361995535


Even though under-the-hood the constraints are being rebuilt from scratch, they will remember if you have flipped them to solve for some other parameter.

To show this, let's flip the constraint for the secondary mass to solve for 'period'.  To do this we need to change the hierarchy back to its original state first.

In [10]:
print("M1: {}, M2: {}, period: {}, q: {}".format(b.get_value(qualifier='mass', component='primary', context='component'),
                                                 b.get_value(qualifier='mass', component='secondary', context='component'),
                                                 b.get_value(qualifier='period', component='binary', context='component'),
                                                 b.get_value(qualifier='q', component='binary', context='component')))

M1: 0.8878338895964281, M2: 1.109792361995535, period: 1.0, q: 0.8


In [11]:
b.set_hierarchy('orbit:binary(star:primary, star:secondary)')

In [12]:
b.flip_constraint('mass@secondary@constraint', 'period')

<ConstraintParameter: {period@binary@component} = ((3.94784176043574320e+01 * ({sma@binary@component} ** 3.000000)) / (({mass@secondary@component} * ((1.00000000000000000e+00 / {q@binary@component}) + 1.00000000000000000e+00)) * 2.94220621750441933e+03)) ** (1./2) (solar units) => 1.0 d>

In [13]:
print("M1: {}, M2: {}, period: {}, q: {}".format(b.get_value(qualifier='mass', component='primary', context='component'),
                                                 b.get_value(qualifier='mass', component='secondary', context='component'),
                                                 b.get_value(qualifier='period', component='binary', context='component'),
                                                 b.get_value(qualifier='q', component='binary', context='component')))

M1: 1.109792361995535, M2: 0.8878338895964281, period: 1.0, q: 0.8


In [14]:
b.set_value(qualifier='mass', component='secondary', context='component', value=1.0)

In [15]:
print("M1: {}, M2: {}, period: {}, q: {}".format(b.get_value(qualifier='mass', component='primary', context='component'),
                                                 b.get_value(qualifier='mass', component='secondary', context='component'),
                                                 b.get_value(qualifier='period', component='binary', context='component'),
                                                 b.get_value(qualifier='q', component='binary', context='component')))

M1: 1.2499999999999996, M2: 1.0, period: 0.9422493776046913, q: 0.8
